<a href="https://colab.research.google.com/github/Holmes-Alan/DM_FM_tutorial/blob/main/DM_FM_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦕 Dinosaur Generative Models — Diffusion & Flow Matching Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/DM_FM_tutorial/blob/main/DM_FM_Tutorial.ipynb)

This notebook is a **hands-on tutorial** for understanding and comparing **9 generative models** on the iconic Datasaurus Dinosaur 2D point-cloud dataset.

| Model | Family | Key Idea |
|-------|--------|----------|
| **VAE** | Latent variable | Encoder–decoder + KL regularisation |
| **DDPM** | Diffusion (SDE) | Predict noise ε at each diffusion step |
| **DDIM** | Diffusion (ODE) | Deterministic sampler on top of DDPM |
| **Score Matching** | Energy-based | Learn ∇ log p(x) via DSM |
| **Consistency Model** | Diffusion (jump) | Directly map any noisy x_t → x_0 |
| **Flow Matching** | Flow (ODE) | Learn straight-line velocity field |
| **Rectified Flow** | Flow (ODE) | Reflow coupling for 1-step generation |
| **MeanFlow** | Flow (mean vel.) | Self-consistent mean velocity field |
| **Drifting Model** | Direct | Mean-shift drift field, 1-NFE sampler |

### Learning outcomes
- Understand the **VAE → DDPM → Score → Flow Matching** progression conceptually.
- Train every model from scratch in Colab (GPU strongly recommended).
- Evaluate with **MMD, Sinkhorn divergence, and Coverage** metrics.
- Visualise sampling trajectories as animations.

> **⚠️ Runtime recommendation:** Use a **GPU runtime** (`Runtime → Change runtime type → T4 GPU`).  
> Training all 9 models takes ~20–40 min on a T4; individual models take 2–5 min each.


## 0 · Setup

Install dependencies, clone the repo, and upload (or auto-download) the data file.

In [ ]:
# ── Install / upgrade dependencies ──────────────────────────────────────────
!pip install -q numpy matplotlib pandas Pillow tqdm

# ── Check GPU ────────────────────────────────────────────────────────────────
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — training will be slower. "
          "Go to Runtime → Change runtime type → T4 GPU.")


In [ ]:
# ── Get the source files ──────────────────────────────────────────────────────
# Option A: clone the GitHub repo (preferred)
# !git clone https://github.com/YOUR_USERNAME/DM_FM_tutorial.git
# %cd DM_FM_tutorial

# Option B: upload train.py + test.py + datasaurus.csv manually, then run:
# from google.colab import files
# uploaded = files.upload()   # upload train.py, test.py, datasaurus.csv

# Option C (used here): download datasaurus.csv from the canonical source
#   and paste train.py / test.py inline (see Section 2 below).

import os, urllib.request

CSV_URL = "https://raw.githubusercontent.com/YassineYousfi/tiny-rf/master/datasaurus.csv"
if not os.path.exists("datasaurus.csv"):
    urllib.request.urlretrieve(CSV_URL, "datasaurus.csv")
    print("✅ datasaurus.csv downloaded.")
else:
    print("✅ datasaurus.csv already present.")


## 1 · Data Exploration — The Datasaurus Dinosaur 🦕

The dataset contains 142 canonical 2D points that trace the outline of a cartoon dinosaur.  
We oversample with jitter to create a realistic point-cloud generative target.

The normalised coordinates lie roughly in x ∈ [−2.4, 3.3], y ∈ [−3.8, 4.3] — a non-trivial,  
**non-Gaussian** distribution that challenges every generative model differently.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Reproduce the data loading logic from train.py ───────────────────────────
def load_dinosaur(n_samples=8000, noise=0.15, seed=42):
    rng = np.random.default_rng(seed)
    df  = pd.read_csv("datasaurus.csv")
    df  = df[df["dataset"] == "dino"]          # 142 canonical points

    ix  = rng.integers(0, len(df), n_samples)
    x   = df["x"].to_numpy()[ix] + rng.normal(size=n_samples) * noise
    y   = df["y"].to_numpy()[ix] + rng.normal(size=n_samples) * noise

    x = (x / 54.0 - 1.0) * 4.0
    y = (y / 48.0 - 1.0) * 4.0
    return np.stack([x, y], axis=1).astype(np.float32)

pts = load_dinosaur()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.suptitle("Datasaurus Dinosaur Point Cloud", fontsize=13, fontweight="bold")

# Left: full cloud
axes[0].scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.3, c="steelblue")
axes[0].set_title(f"Full cloud  (N = {len(pts):,})")
axes[0].set_aspect("equal"); axes[0].grid(alpha=0.3)

# Right: 2-D histogram
axes[1].hist2d(pts[:, 0], pts[:, 1], bins=60, cmap="Blues")
axes[1].set_title("2-D density")
axes[1].set_aspect("equal")

plt.tight_layout()
plt.show()

print(f"x range : [{pts[:,0].min():.2f}, {pts[:,0].max():.2f}]")
print(f"y range : [{pts[:,1].min():.2f}, {pts[:,1].max():.2f}]")
print(f"mean    : {pts.mean(0)}")
print(f"std     : {pts.std(0)}")


## 2 · Model & Training Code

We write `train.py` and `test.py` to disk so that later cells can import from them  
(exactly as the original repo intends).  
> **Skip this section** if you already have `train.py` / `test.py` in your working directory.


In [ ]:
# ── Write train.py to disk ────────────────────────────────────────────────────
# Paste the full content of train.py here, or fetch it from GitHub:

TRAIN_PY_URL = "https://raw.githubusercontent.com/YOUR_USERNAME/DM_FM_tutorial/main/train.py"
TEST_PY_URL  = "https://raw.githubusercontent.com/YOUR_USERNAME/DM_FM_tutorial/main/test.py"

import urllib.request, os

# Uncomment these lines if you are using GitHub:
# urllib.request.urlretrieve(TRAIN_PY_URL, "train.py")
# urllib.request.urlretrieve(TEST_PY_URL,  "test.py")

# ── OR: upload the files manually ────────────────────────────────────────────
# from google.colab import files
# print("Upload train.py:"); files.upload()
# print("Upload test.py:");  files.upload()

# ── Verify files are present ──────────────────────────────────────────────────
for fname in ["train.py", "test.py", "datasaurus.csv"]:
    status = "✅" if os.path.exists(fname) else "❌  MISSING"
    print(f"{status}  {fname}")


> **How to get the files into Colab:**
> 1. **GitHub (recommended):** uncomment the `urllib.request` lines above, replacing `YOUR_USERNAME`.
> 2. **Manual upload:** run the `files.upload()` lines and upload `train.py`, `test.py`, `datasaurus.csv`.
> 3. **Local Colab:** if you mounted Google Drive, adjust paths accordingly.


## 3 · Architecture Overview

All continuous-time models share a common **time-conditioned MLP** backbone:

```
x (2-D) ──┐
           ├──► Linear(2 + t_dim, hidden) ──► SiLU ──► … ──► Linear(hidden, 2)
t ──► SinusoidalEmbed(t_dim) ──┘
```

The sinusoidal embedding maps scalar t → ℝ^{t_dim} using log-spaced frequencies,  
exactly as in the original Transformer positional encoding.

**Schedules define the forward process** x_t = α(t)·x₀ + β(t)·ε:

| Schedule | α(t) | β(t) | Used by |
|----------|------|------|---------|
| DDPM Cosine | √ᾱ_t | √(1−ᾱ_t) | DDPM, DDIM |
| Linear | 1−t | t | Flow Matching, Rectified Flow |
| Trigonometric | cos(πt/2) | sin(πt/2) | MeanFlow, Score |


In [ ]:
# ── Visualise the three schedules ────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(0, 1, 300)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle("Forward Process Schedules  α(t)·x₀ + β(t)·ε", fontsize=12, fontweight="bold")

schedules = {
    "Linear\n(Flow Matching)":     (1 - t, t),
    "Trigonometric\n(MeanFlow)":   (np.cos(np.pi * t / 2), np.sin(np.pi * t / 2)),
}

# DDPM cosine (discrete, approximated for illustration)
s = 0.008
steps = np.arange(1001)
abar = np.cos(((steps / 1000 + s) / (1 + s)) * np.pi / 2) ** 2
abar /= abar[0]
t_ddpm = steps / 1000
alpha_ddpm = np.sqrt(abar)
beta_ddpm  = np.sqrt(1 - abar)

for ax, (name, (a, b)) in zip(axes[:2], schedules.items()):
    ax.plot(t, a, label="α(t)  signal", lw=2, color="steelblue")
    ax.plot(t, b, label="β(t)  noise",  lw=2, color="coral", linestyle="--")
    ax.fill_between(t, 0, a, alpha=0.1, color="steelblue")
    ax.fill_between(t, 0, b, alpha=0.1, color="coral")
    ax.set_title(name); ax.legend(fontsize=9); ax.grid(alpha=0.3)
    ax.set_xlabel("t"); ax.set_xlim(0, 1); ax.set_ylim(-0.05, 1.05)

axes[2].plot(t_ddpm, alpha_ddpm, label="α(t)  signal", lw=2, color="steelblue")
axes[2].plot(t_ddpm, beta_ddpm,  label="β(t)  noise",  lw=2, color="coral", ls="--")
axes[2].fill_between(t_ddpm, 0, alpha_ddpm, alpha=0.1, color="steelblue")
axes[2].fill_between(t_ddpm, 0, beta_ddpm,  alpha=0.1, color="coral")
axes[2].set_title("DDPM Cosine"); axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)
axes[2].set_xlabel("t (normalised)"); axes[2].set_xlim(0, 1); axes[2].set_ylim(-0.05, 1.05)

plt.tight_layout(); plt.show()


## 4 · Train Individual Models

Each cell below trains **one model** and saves a checkpoint to `./checkpoints/<model>/`.  
Run only the models you are interested in — or run **Section 5** to train all at once.

> **Tip:** 3000 epochs ≈ 2–3 min on a T4 GPU for most models.  
> Use 5000–8000 epochs for higher quality results.


### 4.1 · VAE — Variational Autoencoder

The simplest baseline. Learns a 2-D Gaussian latent space via the ELBO.
Samples by drawing z~N(0,I) and decoding. Often blurry due to mode averaging.

In [ ]:
# ── VAE Training ─────────────────────────────────────────────────────
# @title Train VAE { run: "auto" }
MODEL   = "vae"     # @param {"type":"string"}
EPOCHS  = 3000       # @param {"type":"integer"}
LR      = 3e-4       # @param {"type":"number"}
BATCH   = 512        # @param {"type":"integer"}
N_DATA  = 16000      # @param {"type":"integer"}

!python train.py --model {MODEL} --epochs {EPOCHS} --lr {LR} --batch {BATCH} --n_data {N_DATA}


In [ ]:
# ── Visualise VAE training loss ─────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, os

loss_path = f"./checkpoints/vae/train_loss.npy"
if os.path.exists(loss_path):
    losses = np.load(loss_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    fig.suptitle(f"VAE Training Loss", fontweight="bold")

    ax1.plot(losses, lw=1.2, color="steelblue")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Raw loss"); ax1.grid(alpha=0.3)

    # log-scale if useful
    if losses.min() > 0 and losses.max() / losses.min() > 5:
        ax2.plot(losses, lw=1.2, color="coral")
        ax2.set_yscale("log")
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss (log)"); ax2.set_title("Log scale"); ax2.grid(alpha=0.3)
    else:
        window = max(1, len(losses) // 20)
        smooth = np.convolve(losses, np.ones(window)/window, mode="valid")
        ax2.plot(losses, lw=0.8, alpha=0.4, color="steelblue", label="raw")
        ax2.plot(np.arange(window, len(losses)+1), smooth, lw=2, color="steelblue", label="MA-"+str(window))
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.set_title("Smoothed"); ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout(); plt.show()
    print(f"Final loss : {losses[-1]:.6f}")
    print(f"Best  loss : {losses.min():.6f}  (epoch {losses.argmin()+1})")
else:
    print(f"⚠️  No checkpoint found at {loss_path} — did training complete?")


### 4.2 · DDPM — Denoising Diffusion Probabilistic Model

The canonical diffusion model. Trains ε-prediction network on T=1000 noisy steps.
DDIM (below) reuses these weights for faster, deterministic sampling.

In [ ]:
# ── DDPM Training ─────────────────────────────────────────────────────
# @title Train DDPM { run: "auto" }
MODEL   = "ddpm"     # @param {"type":"string"}
EPOCHS  = 3000       # @param {"type":"integer"}
LR      = 3e-4       # @param {"type":"number"}
BATCH   = 512        # @param {"type":"integer"}
N_DATA  = 16000      # @param {"type":"integer"}

!python train.py --model {MODEL} --epochs {EPOCHS} --lr {LR} --batch {BATCH} --n_data {N_DATA}


In [ ]:
# ── Visualise DDPM training loss ─────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, os

loss_path = f"./checkpoints/ddpm/train_loss.npy"
if os.path.exists(loss_path):
    losses = np.load(loss_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    fig.suptitle(f"DDPM Training Loss", fontweight="bold")

    ax1.plot(losses, lw=1.2, color="steelblue")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Raw loss"); ax1.grid(alpha=0.3)

    # log-scale if useful
    if losses.min() > 0 and losses.max() / losses.min() > 5:
        ax2.plot(losses, lw=1.2, color="coral")
        ax2.set_yscale("log")
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss (log)"); ax2.set_title("Log scale"); ax2.grid(alpha=0.3)
    else:
        window = max(1, len(losses) // 20)
        smooth = np.convolve(losses, np.ones(window)/window, mode="valid")
        ax2.plot(losses, lw=0.8, alpha=0.4, color="steelblue", label="raw")
        ax2.plot(np.arange(window, len(losses)+1), smooth, lw=2, color="steelblue", label="MA-"+str(window))
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.set_title("Smoothed"); ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout(); plt.show()
    print(f"Final loss : {losses[-1]:.6f}")
    print(f"Best  loss : {losses.min():.6f}  (epoch {losses.argmin()+1})")
else:
    print(f"⚠️  No checkpoint found at {loss_path} — did training complete?")


### 4.3 · Score Matching — NCSN-style multi-scale DSM

Estimates the score ∇ log p(x) at L noise levels via denoising score matching.
Sampling uses annealed Langevin dynamics from high to low noise.

In [ ]:
# ── Score Matching Training ─────────────────────────────────────────────────────
# @title Train Score Matching { run: "auto" }
MODEL   = "score"     # @param {"type":"string"}
EPOCHS  = 3000       # @param {"type":"integer"}
LR      = 3e-4       # @param {"type":"number"}
BATCH   = 512        # @param {"type":"integer"}
N_DATA  = 16000      # @param {"type":"integer"}

!python train.py --model {MODEL} --epochs {EPOCHS} --lr {LR} --batch {BATCH} --n_data {N_DATA}


In [ ]:
# ── Visualise Score Matching training loss ─────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, os

loss_path = f"./checkpoints/score/train_loss.npy"
if os.path.exists(loss_path):
    losses = np.load(loss_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    fig.suptitle(f"Score Matching Training Loss", fontweight="bold")

    ax1.plot(losses, lw=1.2, color="steelblue")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Raw loss"); ax1.grid(alpha=0.3)

    # log-scale if useful
    if losses.min() > 0 and losses.max() / losses.min() > 5:
        ax2.plot(losses, lw=1.2, color="coral")
        ax2.set_yscale("log")
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss (log)"); ax2.set_title("Log scale"); ax2.grid(alpha=0.3)
    else:
        window = max(1, len(losses) // 20)
        smooth = np.convolve(losses, np.ones(window)/window, mode="valid")
        ax2.plot(losses, lw=0.8, alpha=0.4, color="steelblue", label="raw")
        ax2.plot(np.arange(window, len(losses)+1), smooth, lw=2, color="steelblue", label="MA-"+str(window))
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.set_title("Smoothed"); ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout(); plt.show()
    print(f"Final loss : {losses[-1]:.6f}")
    print(f"Best  loss : {losses.min():.6f}  (epoch {losses.argmin()+1})")
else:
    print(f"⚠️  No checkpoint found at {loss_path} — did training complete?")


### 4.4 · Consistency Model — CT trained from scratch

Trains a consistency function f_θ(x_t, t) → x_0 for any t simultaneously.
Enables 1-step generation; more steps strictly improve quality.

In [ ]:
# ── Consistency Model Training ─────────────────────────────────────────────────────
# @title Train Consistency Model { run: "auto" }
MODEL   = "consistency"     # @param {"type":"string"}
EPOCHS  = 5000       # @param {"type":"integer"}
LR      = 3e-4       # @param {"type":"number"}
BATCH   = 512        # @param {"type":"integer"}
N_DATA  = 16000      # @param {"type":"integer"}

!python train.py --model {MODEL} --epochs {EPOCHS} --lr {LR} --batch {BATCH} --n_data {N_DATA}


In [ ]:
# ── Visualise Consistency Model training loss ─────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, os

loss_path = f"./checkpoints/consistency/train_loss.npy"
if os.path.exists(loss_path):
    losses = np.load(loss_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    fig.suptitle(f"Consistency Model Training Loss", fontweight="bold")

    ax1.plot(losses, lw=1.2, color="steelblue")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Raw loss"); ax1.grid(alpha=0.3)

    # log-scale if useful
    if losses.min() > 0 and losses.max() / losses.min() > 5:
        ax2.plot(losses, lw=1.2, color="coral")
        ax2.set_yscale("log")
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss (log)"); ax2.set_title("Log scale"); ax2.grid(alpha=0.3)
    else:
        window = max(1, len(losses) // 20)
        smooth = np.convolve(losses, np.ones(window)/window, mode="valid")
        ax2.plot(losses, lw=0.8, alpha=0.4, color="steelblue", label="raw")
        ax2.plot(np.arange(window, len(losses)+1), smooth, lw=2, color="steelblue", label="MA-"+str(window))
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.set_title("Smoothed"); ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout(); plt.show()
    print(f"Final loss : {losses[-1]:.6f}")
    print(f"Best  loss : {losses.min():.6f}  (epoch {losses.argmin()+1})")
else:
    print(f"⚠️  No checkpoint found at {loss_path} — did training complete?")


### 4.5 · Flow Matching — Linear interpolant CFM

Learns a velocity field v_θ such that integrating the ODE from ε→x_0.
Target velocity is constant: v = ε - x_0 (straight-line paths).

In [ ]:
# ── Flow Matching Training ─────────────────────────────────────────────────────
# @title Train Flow Matching { run: "auto" }
MODEL   = "flow"     # @param {"type":"string"}
EPOCHS  = 3000       # @param {"type":"integer"}
LR      = 3e-4       # @param {"type":"number"}
BATCH   = 512        # @param {"type":"integer"}
N_DATA  = 16000      # @param {"type":"integer"}

!python train.py --model {MODEL} --epochs {EPOCHS} --lr {LR} --batch {BATCH} --n_data {N_DATA}


In [ ]:
# ── Visualise Flow Matching training loss ─────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, os

loss_path = f"./checkpoints/flow/train_loss.npy"
if os.path.exists(loss_path):
    losses = np.load(loss_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    fig.suptitle(f"Flow Matching Training Loss", fontweight="bold")

    ax1.plot(losses, lw=1.2, color="steelblue")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Raw loss"); ax1.grid(alpha=0.3)

    # log-scale if useful
    if losses.min() > 0 and losses.max() / losses.min() > 5:
        ax2.plot(losses, lw=1.2, color="coral")
        ax2.set_yscale("log")
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss (log)"); ax2.set_title("Log scale"); ax2.grid(alpha=0.3)
    else:
        window = max(1, len(losses) // 20)
        smooth = np.convolve(losses, np.ones(window)/window, mode="valid")
        ax2.plot(losses, lw=0.8, alpha=0.4, color="steelblue", label="raw")
        ax2.plot(np.arange(window, len(losses)+1), smooth, lw=2, color="steelblue", label="MA-"+str(window))
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.set_title("Smoothed"); ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout(); plt.show()
    print(f"Final loss : {losses[-1]:.6f}")
    print(f"Best  loss : {losses.min():.6f}  (epoch {losses.argmin()+1})")
else:
    print(f"⚠️  No checkpoint found at {loss_path} — did training complete?")


### 4.6 · Rectified Flow — 1-RF + 2-RF reflow

Stage-1: standard flow matching (random coupling).
Stage-2 (reflow): retrain on ODE-induced pairs to straighten trajectories.

In [ ]:
# ── Rectified Flow Training ─────────────────────────────────────────────────────
# @title Train Rectified Flow { run: "auto" }
MODEL   = "rectified"     # @param {"type":"string"}
EPOCHS  = 4000       # @param {"type":"integer"}
LR      = 3e-4       # @param {"type":"number"}
BATCH   = 512        # @param {"type":"integer"}
N_DATA  = 16000      # @param {"type":"integer"}

!python train.py --model {MODEL} --epochs {EPOCHS} --lr {LR} --batch {BATCH} --n_data {N_DATA}


In [ ]:
# ── Visualise Rectified Flow training loss ─────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, os

loss_path = f"./checkpoints/rectified/train_loss.npy"
if os.path.exists(loss_path):
    losses = np.load(loss_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    fig.suptitle(f"Rectified Flow Training Loss", fontweight="bold")

    ax1.plot(losses, lw=1.2, color="steelblue")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Raw loss"); ax1.grid(alpha=0.3)

    # log-scale if useful
    if losses.min() > 0 and losses.max() / losses.min() > 5:
        ax2.plot(losses, lw=1.2, color="coral")
        ax2.set_yscale("log")
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss (log)"); ax2.set_title("Log scale"); ax2.grid(alpha=0.3)
    else:
        window = max(1, len(losses) // 20)
        smooth = np.convolve(losses, np.ones(window)/window, mode="valid")
        ax2.plot(losses, lw=0.8, alpha=0.4, color="steelblue", label="raw")
        ax2.plot(np.arange(window, len(losses)+1), smooth, lw=2, color="steelblue", label="MA-"+str(window))
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.set_title("Smoothed"); ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout(); plt.show()
    print(f"Final loss : {losses[-1]:.6f}")
    print(f"Best  loss : {losses.min():.6f}  (epoch {losses.argmin()+1})")
else:
    print(f"⚠️  No checkpoint found at {loss_path} — did training complete?")


### 4.7 · MeanFlow — Self-consistent mean velocity

Predicts mean velocity ū_θ(x_t, t) such that x_0 = x_t - t·ū_θ.
Self-consistency constraint: ū_t + t·∂_t ū_t = instantaneous velocity.

In [ ]:
# ── MeanFlow Training ─────────────────────────────────────────────────────
# @title Train MeanFlow { run: "auto" }
MODEL   = "meanflow"     # @param {"type":"string"}
EPOCHS  = 4000       # @param {"type":"integer"}
LR      = 3e-4       # @param {"type":"number"}
BATCH   = 512        # @param {"type":"integer"}
N_DATA  = 16000      # @param {"type":"integer"}

!python train.py --model {MODEL} --epochs {EPOCHS} --lr {LR} --batch {BATCH} --n_data {N_DATA}


In [ ]:
# ── Visualise MeanFlow training loss ─────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, os

loss_path = f"./checkpoints/meanflow/train_loss.npy"
if os.path.exists(loss_path):
    losses = np.load(loss_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    fig.suptitle(f"MeanFlow Training Loss", fontweight="bold")

    ax1.plot(losses, lw=1.2, color="steelblue")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Raw loss"); ax1.grid(alpha=0.3)

    # log-scale if useful
    if losses.min() > 0 and losses.max() / losses.min() > 5:
        ax2.plot(losses, lw=1.2, color="coral")
        ax2.set_yscale("log")
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss (log)"); ax2.set_title("Log scale"); ax2.grid(alpha=0.3)
    else:
        window = max(1, len(losses) // 20)
        smooth = np.convolve(losses, np.ones(window)/window, mode="valid")
        ax2.plot(losses, lw=0.8, alpha=0.4, color="steelblue", label="raw")
        ax2.plot(np.arange(window, len(losses)+1), smooth, lw=2, color="steelblue", label="MA-"+str(window))
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.set_title("Smoothed"); ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout(); plt.show()
    print(f"Final loss : {losses[-1]:.6f}")
    print(f"Best  loss : {losses.min():.6f}  (epoch {losses.argmin()+1})")
else:
    print(f"⚠️  No checkpoint found at {loss_path} — did training complete?")


### 4.8 · Drifting Model — Mean-shift drifting field

A direct z→x_0 generator trained by the mean-shift drifting field V.
V pushes generated samples toward real data. At equilibrium V→0. **1-NFE**.

In [ ]:
# ── Drifting Model Training ─────────────────────────────────────────────────────
# @title Train Drifting Model { run: "auto" }
MODEL   = "drifting"     # @param {"type":"string"}
EPOCHS  = 4000       # @param {"type":"integer"}
LR      = 3e-4       # @param {"type":"number"}
BATCH   = 512        # @param {"type":"integer"}
N_DATA  = 16000      # @param {"type":"integer"}

!python train.py --model {MODEL} --epochs {EPOCHS} --lr {LR} --batch {BATCH} --n_data {N_DATA}


In [ ]:
# ── Visualise Drifting Model training loss ─────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, os

loss_path = f"./checkpoints/drifting/train_loss.npy"
if os.path.exists(loss_path):
    losses = np.load(loss_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    fig.suptitle(f"Drifting Model Training Loss", fontweight="bold")

    ax1.plot(losses, lw=1.2, color="steelblue")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Raw loss"); ax1.grid(alpha=0.3)

    # log-scale if useful
    if losses.min() > 0 and losses.max() / losses.min() > 5:
        ax2.plot(losses, lw=1.2, color="coral")
        ax2.set_yscale("log")
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss (log)"); ax2.set_title("Log scale"); ax2.grid(alpha=0.3)
    else:
        window = max(1, len(losses) // 20)
        smooth = np.convolve(losses, np.ones(window)/window, mode="valid")
        ax2.plot(losses, lw=0.8, alpha=0.4, color="steelblue", label="raw")
        ax2.plot(np.arange(window, len(losses)+1), smooth, lw=2, color="steelblue", label="MA-"+str(window))
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.set_title("Smoothed"); ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout(); plt.show()
    print(f"Final loss : {losses[-1]:.6f}")
    print(f"Best  loss : {losses.min():.6f}  (epoch {losses.argmin()+1})")
else:
    print(f"⚠️  No checkpoint found at {loss_path} — did training complete?")


## 5 · (Optional) Train ALL Models at Once

This cell loops through every model with recommended epoch counts.  
Expected wall-clock on T4: ~25–40 minutes total.


In [ ]:
# ── Train ALL models sequentially ────────────────────────────────────────────
# @title Train all models {{ run: "auto" }}
import subprocess, time

MODELS_EPOCHS = {
    "vae":         3000,
    "ddpm":        3000,
    "score":       3000,
    "consistency": 5000,
    "flow":        3000,
    "rectified":   4000,
    "meanflow":    4000,
    "drifting":    4000,
}

timings = {}
for model, epochs in MODELS_EPOCHS.items():
    print(f"\n{'='*55}")
    print(f"  Training {model.upper()}  ({epochs} epochs)")
    print(f"{'='*55}")
    t0 = time.time()
    result = subprocess.run(
        ["python", "train.py",
         "--model", model,
         "--epochs", str(epochs),
         "--lr", "3e-4",
         "--batch", "512",
         "--n_data", "16000"],
        capture_output=False
    )
    elapsed = time.time() - t0
    timings[model] = elapsed
    status = "✅" if result.returncode == 0 else "❌"
    print(f"  {status} {model:15s}  {elapsed:.0f}s")

print("\n── Summary ──────────────────────────────────────────")
for m, t in timings.items():
    print(f"  {m:<15}  {t:.0f}s")
print(f"  {'TOTAL':<15}  {sum(timings.values()):.0f}s")


## 6 · Sampling Deep-Dive

### 6.1 · DDPM vs DDIM — Stochastic vs Deterministic Sampling

DDIM reuses the DDPM-trained weights but replaces the stochastic reverse SDE  
with a **deterministic ODE**, enabling high-quality generation with far fewer steps.

| | DDPM | DDIM |
|--|------|------|
| Sampler | Ancestral SDE | Deterministic ODE |
| Steps for good quality | ~200–1000 | ~20–50 |
| Diversity source | Gaussian noise at each step | Only initial noise z₀ |


In [ ]:
# ── Side-by-side: DDPM ancestral vs DDIM deterministic ───────────────────────
import torch, numpy as np, matplotlib.pyplot as plt, sys

# Make sure train.py / test.py are importable from this directory
if "." not in sys.path: sys.path.insert(0, ".")

from train import DDPMModel, DDPMSchedule, load_dinosaur
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Load DDPM checkpoint ─────────────────────────────────────────────────────
import json, os
ckpt_dir = "./checkpoints/ddpm"
if not os.path.exists(f"{ckpt_dir}/model.pt"):
    print("⚠️  DDPM checkpoint not found — run Section 4.2 first.")
else:
    with open(f"{ckpt_dir}/config.json") as f:
        cfg = json.load(f)
    T = cfg.get("T_ddpm", 1000)

    model = DDPMModel(hidden=256, depth=4).to(device)
    model.load_state_dict(torch.load(f"{ckpt_dir}/model.pt", map_location=device))
    model.eval()

    sched = DDPMSchedule(T)
    sched.abar  = sched.abar.to(device)
    sched.betas = sched.betas.to(device)
    sched.alphas = sched.alphas.to(device)

    real_pts = load_dinosaur(n_samples=2000)
    N = 2000

    step_counts = [10, 50, 200, 1000]

    fig, axes = plt.subplots(2, len(step_counts), figsize=(14, 7))
    fig.suptitle("DDPM (top) vs DDIM (bottom) — varying sampling steps", fontsize=13, fontweight="bold")

    for col, S in enumerate(step_counts):
        # ── DDPM ancestral ──────────────────────────────────────────────────
        with torch.no_grad():
            ts = torch.linspace(T, 1, S, dtype=torch.long, device=device)
            x = torch.randn(N, 2, device=device)
            for t_val in ts:
                tb = t_val.expand(N)
                ep = model(x, tb, T=T)
                a  = sched.alphas[t_val - 1]
                ab = sched.abar[t_val]
                ab_prev = sched.abar[t_val - 1] if t_val > 1 else torch.tensor(1.0, device=device)
                beta = sched.betas[t_val - 1]
                mu = (1 / a.sqrt()) * (x - beta / (1 - ab).sqrt() * ep)
                if t_val > 1:
                    var = (1 - ab_prev) / (1 - ab) * beta
                    x = mu + var.sqrt() * torch.randn_like(x)
                else:
                    x = mu
        ddpm_pts = x.cpu().numpy()

        # ── DDIM deterministic ──────────────────────────────────────────────
        with torch.no_grad():
            ts2 = torch.linspace(T - 1, 1, S, dtype=torch.long, device=device)
            x = torch.randn(N, 2, device=device)
            for i, t_val in enumerate(ts2):
                tb = t_val.expand(N)
                ep = model(x, tb, T=T)
                ab_t = sched.abar[t_val].clamp(min=1e-8)
                ab_prev = sched.abar[ts2[i+1]] if i+1 < len(ts2) else torch.tensor(1.0, device=device)
                x0p = (x - (1 - ab_t).sqrt() * ep) / ab_t.sqrt()
                x0p = x0p.clamp(-10, 10)
                x = ab_prev.sqrt() * x0p + (1 - ab_prev).sqrt() * ep
        ddim_pts = x.cpu().numpy()

        xlo, xhi = real_pts[:,0].min()-0.3, real_pts[:,0].max()+0.3
        ylo, yhi = real_pts[:,1].min()-0.3, real_pts[:,1].max()+0.3

        for row, (pts, label, color) in enumerate([
            (ddpm_pts, "DDPM", "steelblue"),
            (ddim_pts, "DDIM", "coral"),
        ]):
            ax = axes[row][col]
            ax.scatter(pts[:,0], pts[:,1], s=3, alpha=0.4, c=color)
            ax.set_xlim(xlo, xhi); ax.set_ylim(ylo, yhi)
            ax.set_aspect("equal"); ax.grid(alpha=0.2)
            ax.set_title(f"{label}  |  {S} steps", fontsize=9)

    plt.tight_layout(); plt.show()


### 6.2 · Flow Matching — Sampling Trajectory

Flow matching integrates dx/dt = v_θ(x_t, t) from t=1 (noise) to t=0 (data).  
With the linear interpolant the trajectory is **straight** in expectation — fewer steps needed.


In [ ]:
# ── Visualise Flow Matching ODE trajectory for a small set of points ──────────
import torch, numpy as np, matplotlib.pyplot as plt, os, sys
if "." not in sys.path: sys.path.insert(0, ".")

from train import FlowModel, load_dinosaur

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = "./checkpoints/flow/model.pt"

if not os.path.exists(ckpt):
    print("⚠️  Flow checkpoint not found — run Section 4.5 first.")
else:
    model = FlowModel(hidden=256, depth=4).to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()

    n_traj = 300      # number of trajectories to visualise
    S      = 50       # integration steps

    with torch.no_grad():
        x = torch.randn(n_traj, 2, device=device)
        traj = [x.cpu().numpy().copy()]
        dt = -1.0 / S
        for i in range(S):
            t_val = 1.0 - i / S
            t = torch.full((n_traj,), t_val, device=device)
            x = x + model(x, t) * dt
            if i % 5 == 0:
                traj.append(x.cpu().numpy().copy())
        traj.append(x.cpu().numpy().copy())

    real_pts = load_dinosaur(n_samples=2000)
    xlo, xhi = real_pts[:,0].min()-0.3, real_pts[:,0].max()+0.3
    ylo, yhi = real_pts[:,1].min()-0.3, real_pts[:,1].max()+0.3

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Flow Matching — ODE Trajectory", fontsize=13, fontweight="bold")

    # Left: trajectory lines
    ax = axes[0]
    colors = plt.cm.viridis(np.linspace(0, 1, n_traj))
    for p in range(0, n_traj, 3):
        pts_seq = np.array([step[p] for step in traj])
        ax.plot(pts_seq[:,0], pts_seq[:,1], lw=0.5, alpha=0.6,
                color=colors[p])
    ax.scatter(traj[0][:,0],  traj[0][:,1],  s=8, c="red",   zorder=5, label="t=1 (noise)")
    ax.scatter(traj[-1][:,0], traj[-1][:,1], s=8, c="green", zorder=5, label="t=0 (data)")
    ax.set_xlim(xlo, xhi); ax.set_ylim(ylo, yhi); ax.set_aspect("equal")
    ax.legend(fontsize=9); ax.grid(alpha=0.25)
    ax.set_title("Sample trajectories (colour = particle index)")

    # Right: final vs real
    ax2 = axes[1]
    ax2.scatter(real_pts[:,0],  real_pts[:,1],  s=3, alpha=0.3, c="coral",     label="Real")
    ax2.scatter(traj[-1][:,0],  traj[-1][:,1],  s=6, alpha=0.7, c="steelblue", label="Generated")
    ax2.set_xlim(xlo, xhi); ax2.set_ylim(ylo, yhi); ax2.set_aspect("equal")
    ax2.legend(fontsize=9); ax2.grid(alpha=0.25)
    ax2.set_title("Final samples vs. ground truth")

    plt.tight_layout(); plt.show()


## 7 · Evaluation Metrics

We use three complementary metrics (all **lower = better**):

| Metric | Formula | What it measures |
|--------|---------|-----------------|
| **MMD** | E[K(x,x)] + E[K(y,y)] − 2E[K(x,y)] | Distribution matching in RKHS (RBF kernel) |
| **Sinkhorn** | regularised Wasserstein-1 | Geometric distance between distributions |
| **Coverage miss** | 1 − Coverage | Fraction of real points NOT covered by generated samples |

### 7.1 · Evaluate a Single Model


In [ ]:
# ── Evaluate one model across sampling step counts ───────────────────────────
# @title Evaluate model {{ run: "auto" }}
MODEL = "flow"  # @param ["vae","ddpm","ddim","score","consistency","flow","rectified","meanflow","drifting"]

!python test.py --model {MODEL} --n_samples 1000

# ── Display saved plots ───────────────────────────────────────────────────────
from IPython.display import Image, display
import os

for fname in ["final_samples.png", "metrics_vs_steps.png"]:
    path = f"./results/{MODEL}/{fname}"
    if os.path.exists(path):
        display(Image(path))


### 7.2 · Evaluate ALL Models & Generate Comparison Table

In [ ]:
# ── Evaluate all trained models ───────────────────────────────────────────────
# This will only evaluate models for which a checkpoint exists.
!python test.py --model all --n_samples 1000

from IPython.display import Image, display
import os

for fname in ["comparison_table.png", "all_models_samples.png", "train_losses.png"]:
    path = f"./results/{fname}"
    if os.path.exists(path):
        print(f"\n── {fname} ──────────────────────────────────────")
        display(Image(path))


## 8 · Sampling Animation (GIF)

Visualise how sample quality **improves with more steps** for any model.

> Uses `--gif` flag in `test.py`. Requires `Pillow` (pre-installed in Colab).
> **Note:** use `IPython.display.HTML` (not `Image`) to see the full animation.


In [ ]:
# ── Generate and display GIF for a chosen model ───────────────────────────────
# @title Animate sampling steps {{ run: "auto" }}
MODEL = "flow"  # @param ["ddpm","ddim","flow","rectified","consistency","meanflow","score"]

!python test.py --model {MODEL} --gif --fps 3 --n_samples 2000

from IPython.display import HTML, display
import os

gif_path = f"./results/{MODEL}/sampling_steps.gif"
if os.path.exists(gif_path):
    # Use HTML — IPython.display.Image shows only the first frame
    display(HTML(f'<img src="{gif_path}" style="max-width:750px; border-radius:8px">'))
else:
    print(f"⚠️  GIF not found at {gif_path}. Did training + evaluation complete?")


## 9 · Conceptual Summary & Takeaways

### The VAE → Diffusion → Flow progression

```
VAE              Deterministic encoder + stochastic latent
  ↓ add time
DDPM             Iterative noise → data via learned ε-prediction (SDE)
  ↓ same network, ODE sampler
DDIM             Deterministic, fewer steps, same weights
  ↓ continuous-time formulation
Score Matching   Learn ∇ log p directly, Langevin sampling
  ↓ straight-line interpolant
Flow Matching    Learn constant velocity ε − x₀ (ODE)
  ↓ reflow coupling
Rectified Flow   Straighten trajectories → approach 1-step
  ↓ mean velocity
MeanFlow         Predict average velocity for 1-step shortcut
  ↓ direct generator + drift field
Drifting Model   No ODE at all — direct z → x₀ with 1-NFE
```

### Key design trade-offs

| Aspect | Slow & high quality | Fast (1–5 steps) |
|--------|--------------------|--------------------|
| **NFEs needed** | DDPM (200–1000) | Consistency, MeanFlow, Drifting |
| **Training stability** | VAE, Flow | Consistency (CT) |
| **Trajectory geometry** | Random (DDPM SDE) | Straight (Rectified Flow) |
| **Model family** | Score-based SDE | Flow / direct generator |

### Further reading
- Ho et al. 2020 — DDPM
- Song et al. 2020 — Score-based generative models
- Song et al. 2021 — DDIM
- Lipman et al. 2022 — Flow Matching
- Liu et al. 2022 — Rectified Flow
- Song et al. 2023 — Consistency Models
- Geng et al. 2024 — MeanFlow
- Deng et al. 2026 — Drifting Model (ICML)


## 10 · Exercises for Students 🎓

Try the following to deepen your understanding:

1. **Vary the VAE β:** retrain with `--beta_vae 0.1`, `1.0`, `4.0`.  
   How does the sample quality and latent space geometry change?

2. **DDPM step budget:** evaluate DDPM at 1, 10, 50, 200, 1000 steps.  
   At what step count does quality plateau? Compare with DDIM.

3. **Flow Matching convergence:** run `--epochs 500`, `1000`, `3000`, `8000`.  
   Plot MMD vs. epochs. Does more training help significantly?

4. **Consistency Model curriculum:** the CT loss *increases* during training  
   because N grows (harder task). Add a `print(N)` statement and confirm this.  
   Does the final *sample quality* still improve?

5. **One-step generation comparison:** compare VAE, Consistency (1 step),  
   MeanFlow (1 step), and Drifting Model. Which captures the dinosaur shape best  
   with a **single forward pass**?

6. **Rectified flow reflow:** visualise sample trajectories from stage-1 RF  
   vs stage-2 RF (after reflow). Are the paths straighter? Does this improve  
   1-step generation?

7. *(Advanced)* Implement a **2nd-order ODE solver (Heun)** for Flow Matching  
   and compare MMD vs. Euler at the same NFE budget.
